# Brain Tumor Detection with YOLO11s

This notebook covers the complete workflow for training and evaluating a YOLO11s model on a labeled MRI brain tumor dataset.

**Classes**
- Glioma
- Meningioma
- No Tumor
- Pituitary

> **Security note:** The Roboflow API key is intentionally not stored in this notebook. Set it as the `ROBOFLOW_API_KEY` environment variable before running the dataset download cell.


## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics roboflow pyyaml pillow opencv-python matplotlib

## 2. Download the Dataset from Roboflow

In [ ]:
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")

if not ROBOFLOW_API_KEY:
    raise ValueError(
        "ROBOFLOW_API_KEY is not set. "
        "Set it as an environment variable before running this cell."
    )

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = rf.workspace("mango-qoesz").project(
    "labeled-mri-brain-tumor-dataset-l8ayj"
)

version = project.version(1)
dataset = version.download("yolov11")

print("Dataset location:", dataset.location)
DATASET_DIR = os.path.abspath(dataset.location)


## 3. Dataset Structure and Summary

In [ ]:
from pathlib import Path

DATASET_DIR = Path(DATASET_DIR)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLITS = ("train", "valid", "test")

print("Dataset path:", DATASET_DIR)
print("\n========== DATASET SUMMARY ==========")

total_images = 0
total_labels = 0

for split in SPLITS:
    images_dir = DATASET_DIR / split / "images"
    labels_dir = DATASET_DIR / split / "labels"

    images = [
        p for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ] if images_dir.exists() else []

    labels = list(labels_dir.glob("*.txt")) if labels_dir.exists() else []

    print(f"\n{split.upper()}")
    print("Images :", len(images))
    print("Labels :", len(labels))

    total_images += len(images)
    total_labels += len(labels)

print("\nTOTAL IMAGES:", total_images)
print("TOTAL LABELS:", total_labels)


## 4. Dataset Configuration

In [ ]:
import yaml

yaml_path = DATASET_DIR / "data.yaml"

if yaml_path.exists():
    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    print("========== DATA.YAML ==========")
    print(yaml.safe_dump(data, sort_keys=False))
else:
    raise FileNotFoundError(f"data.yaml not found: {yaml_path}")


## 5. Label Quality Analysis

In [ ]:
from collections import Counter

CLASS_NAMES = {
    0: "Glioma",
    1: "Meningioma",
    2: "No Tumor",
    3: "Pituitary",
}

empty_labels = []
invalid_labels = []
invalid_boxes = []
class_counter = Counter()
box_counter = Counter()

for split in SPLITS:
    label_dir = DATASET_DIR / split / "labels"

    if not label_dir.exists():
        continue

    for label_file in label_dir.glob("*.txt"):
        with open(label_file, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        if not lines:
            empty_labels.append(str(label_file))
            continue

        box_counter[len(lines)] += 1

        for line in lines:
            values = line.split()

            if len(values) != 5:
                invalid_labels.append((str(label_file), line))
                continue

            try:
                cls, x, y, w, h = map(float, values)
                cls = int(cls)
            except ValueError:
                invalid_labels.append((str(label_file), line))
                continue

            if cls not in CLASS_NAMES:
                invalid_labels.append((str(label_file), line))
                continue

            if not (
                0 <= x <= 1 and
                0 <= y <= 1 and
                0 < w <= 1 and
                0 < h <= 1
            ):
                invalid_boxes.append((str(label_file), line))
                continue

            class_counter[cls] += 1

print("========== LABEL QUALITY ==========")
print("Empty labels   :", len(empty_labels))
print("Invalid labels :", len(invalid_labels))
print("Invalid boxes  :", len(invalid_boxes))

print("\n========== CLASS DISTRIBUTION ==========")
for class_id, name in CLASS_NAMES.items():
    print(f"{class_id} - {name:<12}: {class_counter[class_id]} boxes")


## 6. Check for Duplicate Images

In [ ]:
import hashlib
from collections import defaultdict

def get_image_hashes(folder):
    hashes = defaultdict(list)

    if not folder.exists():
        return hashes

    for image_path in folder.iterdir():
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        with open(image_path, "rb") as f:
            image_hash = hashlib.md5(f.read()).hexdigest()

        hashes[image_hash].append(image_path)

    return hashes

all_hashes = {}

for split in SPLITS:
    image_dir = DATASET_DIR / split / "images"
    all_hashes[split] = get_image_hashes(image_dir)

print("========== CROSS-SPLIT DUPLICATES ==========")

split_names = list(all_hashes)

for i in range(len(split_names)):
    for j in range(i + 1, len(split_names)):
        s1, s2 = split_names[i], split_names[j]
        common = set(all_hashes[s1]) & set(all_hashes[s2])
        print(f"{s1} vs {s2}: {len(common)} duplicate groups")


## 7. Image Quality Analysis

In [ ]:
from PIL import Image
import numpy as np

bad_images = []
small_images = []
grayscale_images = []
widths, heights = [], []

for split in SPLITS:
    image_dir = DATASET_DIR / split / "images"

    if not image_dir.exists():
        continue

    for image_path in image_dir.iterdir():
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            with Image.open(image_path) as img:
                img.verify()

            with Image.open(image_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)

                if w < 150 or h < 150:
                    small_images.append(str(image_path))

                if img.mode == "L":
                    grayscale_images.append(str(image_path))

        except Exception as exc:
            bad_images.append((str(image_path), str(exc)))

print("========== IMAGE QUALITY REPORT ==========")
print("Images analyzed :", len(widths))
print("Bad images      :", len(bad_images))
print("Very small      :", len(small_images))
print("Grayscale       :", len(grayscale_images))

if widths:
    print("\nWidth  - min:", min(widths), "max:", max(widths), "mean:", round(np.mean(widths), 2))
    print("Height - min:", min(heights), "max:", max(heights), "mean:", round(np.mean(heights), 2))


## 8. Visualize Labeled MRI Samples

In [ ]:
import cv2
import random
import matplotlib.pyplot as plt

train_images_dir = DATASET_DIR / "train" / "images"
train_labels_dir = DATASET_DIR / "train" / "labels"

images = [
    p for p in train_images_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

samples = random.sample(images, min(12, len(images)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for ax in axes:
    ax.axis("off")

for ax, image_path in zip(axes, samples):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]
    label_path = train_labels_dir / f"{image_path.stem}.txt"

    if label_path.exists():
        with open(label_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        for line in lines:
            values = line.strip().split()
            if len(values) != 5:
                continue

            cls, xc, yc, bw, bh = map(float, values)
            cls = int(cls)

            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)

            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(
                image,
                CLASS_NAMES.get(cls, str(cls)),
                (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 0, 0),
                2,
            )

    ax.imshow(image)
    ax.set_title(image_path.name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 9. Prepare YOLO Data Configuration

In [ ]:
data = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 4,
    "names": list(CLASS_NAMES.values()),
}

yaml_path = DATASET_DIR / "data.yaml"

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(yaml.safe_dump(data, sort_keys=False))


## 10. Train YOLO11s

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    patience=20,
    device=0,
    name="Brain_Tumor_YOLO11s",
)


## 11. Evaluate the Trained Model on the Test Set

In [ ]:
from ultralytics import YOLO

best_model_path = Path("runs/detect/Brain_Tumor_YOLO11s/weights/best.pt")

if not best_model_path.exists():
    raise FileNotFoundError(f"Trained model not found: {best_model_path}")

best_model = YOLO(str(best_model_path))

test_results = best_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    save_json=True,
)

print("Test evaluation completed.")


## 12. Output Files

After training, the main artifacts are available under:

```text
runs/
└── detect/
    └── Brain_Tumor_YOLO11s/
        ├── weights/
        │   ├── best.pt
        │   └── last.pt
        ├── results.csv
        ├── results.png
        └── ...
```

